In [5]:
import re
from collections import defaultdict
import sys
sys.path.append('..')
from utils.parser import str_to_triplet
from kg_connector.kg_connector import KGConnector
from embeddings.relation_embedder import RelationEmbedder
from typing import Optional

class GroupNDecompose:

    def __init__(self, relation_embedder: Optional[RelationEmbedder] = None, kg_connector: Optional[KGConnector] = None):
        self.relation_embedder : RelationEmbedder = RelationEmbedder() if relation_embedder is None else relation_embedder
        self.kg_connector     : KGConnector      = KGConnector() if kg_connector is None else kg_connector


    def is_unknown(self, entity: str) -> bool:
        if entity is None:
            raise ValueError("Entity cannot be None")
        if not isinstance(entity, str):
            raise TypeError("Entity must be a string")
        if not entity:
            raise ValueError("Entity cannot be an empty string")

        return entity.startswith("unknown_")

    def group_n_decompose(self, triplets: list[str], kg_ref: dict[str, str]):
        """
        Group and decompose triplets based on entity types.
        """

        parsed : list[dict[str, str]] = []
        for triplet in triplets:
            head, relation, tail = str_to_triplet(triplet)
            parsed.append({
                "head"    : head,
                "relation": relation,
                "tail"    : tail,
            })

        completed   : list = []
        incompleted : list = []

        # Separate completed and incompleted triplets
        for triplet in parsed:
            if self.is_unknown(triplet["head"]) or self.is_unknown(triplet["tail"]):
                incompleted.append(triplet)
            else:
                completed.append(triplet)

        # Grouping incompleted triplets by unknown entity
        groups = defaultdict(list)
        for t in incompleted:
            if self.is_unknown(t["head"]):
                groups[t["head"]].append(t)
            elif self.is_unknown(t["tail"]):
                groups[t["tail"]].append(t)

        grouped = {}
        for unk, arr in groups.items():
            explicit_entities   = [] # explicit entities in the group that linked to unknown_
            relations           = []

            for pos, t in arr:
                if pos == "head":
                    explicit_entities.append(t["tail"])
                    relations.append(t["relation"])
                elif pos == "tail":
                    explicit_entities.append(t["head"])
                    relations.append(t["relation"])

            grouped[unk] = {
                "unknown": unk,
                "explicit_entities": explicit_entities,
                "relations": relations,
                "raw_triplets": [x[1] for x in arr],
            }

        # Decompose groups based on entity types
        decomposed = []
        if kg_ref is not None:
            unknown_counter = 999  # sinh unknown mới
            for t in completed:
                e1, r, e2 = t["head"], t["rel"], t["tail"]

                if (e1, e2) not in kg_ref:
                    new_unk = f"unknown_{unknown_counter}"
                    unknown_counter += 1

                    # tách thành 2 incomplete triplets
                    t1 = {"head": e1, "rel": r, "tail": new_unk}
                    t2 = {"head": new_unk, "rel": r, "tail": e2}

                    decomposed.append(t1)
                    decomposed.append(t2)
                else:
                    decomposed.append(t)
        else:
            decomposed = completed[:]  # giữ nguyên nếu không check KG

        return {
            "complete_triplets": completed,
            "incomplete_groups": grouped,
            "decomposed_triplets": decomposed
        }

    def normalize_relation(self, triplet_dicts: list[dict[str, str]]):
        """
        Normalize relations in triplet dictionaries using RelationEmbedder.
        """
        for triplet in triplet_dicts:
            raw_relation = triplet["relation"]
            matched_relation, score = self.relation_embedder.match_relation(raw_relation, top_k=1)[0]
            triplet["relation"] = matched_relation
        return triplet_dicts

    def build_kg_adjacency(self, triplet_dicts: list[dict[str, str]]):
        """
        Build KG adjacency list from triplet dictionaries.
        """
        explicit_entities = set()
        for triplet in triplet_dicts:
            if not self.is_unknown(triplet["head"]):
                explicit_entities.add(triplet["head"])
            if not self.is_unknown(triplet["tail"]):
                explicit_entities.add(triplet["tail"])
        explicit_entities = list(explicit_entities)

        relations = [triplet["relation"] for triplet in triplet_dicts]

        adjacency_list = self.kg_connector.build_local_adj(explicit_entities, relations)
        return adjacency_list


In [6]:
# Test

gnd = GroupNDecompose()
triplets = [
    "<e>Pho</e>|| country || <e>Vietnam</e>",
    "<e>unknown_0</e>|| ingredient || <e>Pho</e>",
    "<e>unknown_0</e>|| ingredient || <e>Mì Quảng</e>",
    "<e>unknown_0</e>|| ingredient || <e>Meat and potato pie</e>"
]
triplet_dicts = [
    {"head": "Pho", "relation": "country", "tail": "Vietnam"},
    {"head": "unknown_0", "relation": "ingredient", "tail": "Pho"},
    {"head": "unknown_0", "relation": "ingredient", "tail": "Mì Quảng"},
    {"head": "unknown_0", "relation": "ingredient", "tail": "Meat and potato pie"}
]
relations = ["country", "ingredient"]
kg_adj = gnd.build_kg_adjacency(triplet_dicts)
print(kg_adj)

Loaded 527 relations and their embeddings from D:\\claimpkg\\claimpkg-clone\\src\\resources\\embeddings.
{'Pho': {'country': ['Vietnam', '"Vietnam"'], 'ingredient': ['Beef', 'Chicken (food)', 'Rice noodles', 'Beef']}, 'Mì Quảng': {'country': ['Vietnam'], 'ingredient': ['Pork', 'Beef', 'Rice noodles', 'Shrimp', 'Turmeric', 'Chicken (food)', 'Beef']}, 'Meat and potato pie': {'country': ['England'], 'ingredient': ['Beef']}, 'Vietnam': {}}
